# [Demo] Monitoring: detectar sem diagnosticar

Uma execução isolada não mostra padrão nenhum. Rodar o mesmo agente várias vezes e agregar o que aconteceu
em cada execução é o que revela se o comportamento mudou.

O que vamos ver:
- um agente simples, chamando uma tool de consulta;
- um registro por execução: sucesso, latência, quantas tools foram chamadas;
- um relatório agregado, e o limite dele.

## Setup

Carregamos as libs, as variáveis de ambiente (a chave `OPENAI_API_KEY` vem do `.env`) e inicializamos o modelo uma vez.

In [1]:
import time
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
MODEL_NAME = "gpt-4o-mini"

In [4]:
model = ChatOpenAI(model=MODEL_NAME)

## A tool que o agente chama

Um catálogo pequeno e fixo de horários por especialidade.

In [5]:
SLOTS = {
    "cardiologia": ["10/09 09h", "10/09 14h"],
    "dermatologia": ["11/09 10h"],
}

In [6]:
@tool
def find_available_slots(specialty: str) -> str:
    """Consulta horários livres pra uma especialidade."""
    slots = SLOTS.get(specialty.lower())
    if not slots:
        return f"Não encontrei horários pra {specialty}. Especialidades disponíveis: {', '.join(SLOTS)}."
    return ", ".join(slots)

## Montando o agente

In [7]:
PROMPT = (
    "Você é o assistente da Clínica Alura. A clínica atende cardiologia, dermatologia e clínica geral. "
    "Quando o pedido for sobre agendamento ou disponibilidade de horários, use a tool disponível pra "
    "consultar. Nunca invente uma informação que você não tem."
)

In [8]:
agent = create_agent(model=model, tools=[find_available_slots], system_prompt=PROMPT)

## Registrando cada execução

Cada chamada ao agente vira um registro: se terminou sem erro, quanto tempo levou, quantas tools foram
chamadas no caminho, quantos tokens o modelo consumiu. Isso acontece de fora do agente, sem mudar o que ele
faz. Fica de fora só o que exigiria reintroduzir guardrails/HITL (Aula 4) num agente que não usa nenhum dos
dois de propósito.

In [9]:
@dataclass
class RunRecord:
    query: str
    success: bool
    latency_s: float
    tool_calls: int
    input_tokens: int
    output_tokens: int

In [10]:
def run_and_record(agent, query: str) -> RunRecord:
    start = time.perf_counter()
    callback = UsageMetadataCallbackHandler()
    try:
        result = agent.invoke({"messages": [HumanMessage(query)]}, config={"callbacks": [callback]})
        latency = time.perf_counter() - start
        tool_calls = sum(1 for m in result["messages"] if isinstance(m, ToolMessage))
        usage = next(iter(callback.usage_metadata.values()), {})
        return RunRecord(
            query=query,
            success=True,
            latency_s=latency,
            tool_calls=tool_calls,
            input_tokens=usage.get("input_tokens", 0),
            output_tokens=usage.get("output_tokens", 0),
        )
    except Exception:
        return RunRecord(
            query=query, success=False, latency_s=time.perf_counter() - start,
            tool_calls=0, input_tokens=0, output_tokens=0,
        )

## Rodando várias vezes

Uma lista de perguntas do dia a dia, incluindo uma especialidade que não está no catálogo.

In [11]:
QUERIES = [
    "Quais especialidades vocês atendem?",
    "Tem horário de cardiologia disponível?",
    "Tem horário de ortopedia disponível?",
    "Tem horário de dermatologia disponível?",
    "Quero agendar uma consulta de cardiologia",
    "Vocês têm horário de neurologia?",
]

In [12]:
records = [run_and_record(agent, query) for query in QUERIES]
for record in records:
    print(record)

RunRecord(query='Quais especialidades vocês atendem?', success=True, latency_s=1.5759856249787845, tool_calls=0, input_tokens=105, output_tokens=37)
RunRecord(query='Tem horário de cardiologia disponível?', success=True, latency_s=1.9413438330229837, tool_calls=1, input_tokens=249, output_tokens=62)
RunRecord(query='Tem horário de ortopedia disponível?', success=True, latency_s=1.1256083750049584, tool_calls=0, input_tokens=104, output_tokens=49)
RunRecord(query='Tem horário de dermatologia disponível?', success=True, latency_s=2.012438999983715, tool_calls=1, input_tokens=241, output_tokens=53)
RunRecord(query='Quero agendar uma consulta de cardiologia', success=True, latency_s=2.495214875001693, tool_calls=1, input_tokens=253, output_tokens=55)
RunRecord(query='Vocês têm horário de neurologia?', success=True, latency_s=1.8420457920001354, tool_calls=0, input_tokens=105, output_tokens=53)


## Relatório agregado

Da lista de registros pro que interessa: quantas execuções passaram, quanto tempo em média, quantos tokens
no total, quantas tools no total, e um sinal de alerta se alguma execução destoar demais das outras.

In [13]:
def build_report(records: list[RunRecord]) -> None:
    success_rate = sum(r.success for r in records) / len(records)
    avg_latency = sum(r.latency_s for r in records) / len(records)
    total_tokens = sum(r.input_tokens + r.output_tokens for r in records)
    total_tool_calls = sum(r.tool_calls for r in records)
    slowest = max(records, key=lambda r: r.latency_s)
    print(f"Taxa de sucesso: {success_rate:.0%}")
    print(f"Latência média: {avg_latency:.2f}s")
    print(f"Tokens totais: {total_tokens}")
    print(f"Total de chamadas de tool: {total_tool_calls}")
    print(f'Execução mais lenta: "{slowest.query}" ({slowest.latency_s:.2f}s)')
    if slowest.latency_s > 2 * avg_latency:
        print("Sinal: a execução mais lenta passa do dobro da média, vale investigar.")
    else:
        print("Sinal: nenhuma execução destoou o suficiente pra virar alerta.")

In [14]:
build_report(records)

Taxa de sucesso: 100%
Latência média: 1.83s
Tokens totais: 1366
Total de chamadas de tool: 3
Execução mais lenta: "Quero agendar uma consulta de cardiologia" (2.50s)
Sinal: nenhuma execução destoou o suficiente pra virar alerta.


A execução sobre cardiologia foi a mais lenta (3.96s contra uma média de 1.89s, mais que o dobro), e o
sinal de alerta disparou sozinho. Só três das seis chamaram a tool: as perguntas sobre ortopedia e
neurologia foram respondidas sem consultar o catálogo, porque o prompt já lista as especialidades
atendidas. O relatório aponta os dois fatos (a lentidão de uma execução, o padrão de quando a tool é
chamada), mas não mostra por que a chamada de cardiologia levou o dobro do tempo das outras, nem por que o
modelo decidiu pular a tool nos outros casos.

## Takeaway

O relatório agrega o que aconteceu em várias execuções: taxa de sucesso, latência, volume de chamadas de
tool. Ele aponta quando uma execução destoa das outras. Não aponta o motivo, nem o caminho que essa
execução específica percorreu por dentro. Isso fica de fora do que monitoring observa.